# hyrax-parsnip workflow demo

This notebook walks through the full workflow of running [ParSNIP](https://parsnip.readthedocs.io)
through [Hyrax](https://hyrax.readthedocs.io) on light curves stored in a
[HATS](https://hats.readthedocs.io) catalog:

1. **Simulate** SN Ia, II and Ib/c light curves on the real Rubin DP2 visits with
   [lightcurvelynx](https://github.com/lincc-frameworks/lightcurvelynx), and write them as a HATS catalog in the
   **same schema as the DP2 DIA catalog** (`diaObject` rows with nested `diaObjectForcedSource` photometry in nJy).
2. Run **inference with the pretrained `plasticc` ParSNIP model**, with no training needed, and with
   `plasticc_photoz`, which doesn't need redshifts (the setup for real data).
3. Look at the ParSNIP latent space.
4. **Train a LightGBM classifier** on the latents, save it, reload it, and classify objects.
5. *(Optional)* **Fine-tune** ParSNIP with `h.train()`, export it, and run inference again.

The simulation follows `dp2_sims_visitdetector.ipynb` in
`lightcurvelynx_rubin_dp2`. Because the simulated catalog has the
real-data schema, everything after section 1 runs unchanged on the DP2 DIA catalog
(see `examples/parsnip_classify_rubin_dia.py`).

Outputs go to `data/` and `results/`, which git ignores.

**Environment (NERSC).** lightcurvelynx comes from `desc-td-env`, so build the venv on top of it:

```bash
/global/common/software/lsst/install/td_env/2026-09-03-32-23/py/envs/td_env/bin/python \
    -m venv --system-site-packages .venv
.venv/bin/pip install -e ".[notebook]"
.venv/bin/python -m ipykernel install --user --name hyrax-parsnip-td --display-name "hyrax-parsnip (td_env)"
```

and run this notebook with the `hyrax-parsnip (td_env)` kernel.

In [ ]:
from pathlib import Path

import lsdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hyrax import Hyrax

import hyrax_parsnip
import simulate_dp2

DATA_DIR = Path("data")
RESULTS_DIR = Path("results") / "demo"
CATALOG_PATH = DATA_DIR / "dp2_sim_sne"

## 1. Simulate DP2 light curves as a HATS catalog

`hyrax-parsnip` reads a **nested** HATS catalog with one row per object. We write the simulations with the
column names of the DP2 DIA catalog, plus truth columns:

| column                                   | type                                               |
|------------------------------------------|----------------------------------------------------|
| `diaObjectId`, `ra`, `dec`, `nDiaSources` | per object, as in DP2                             |
| `diaObjectForcedSource`                  | nested: `midpointMjdTai`, `band`, `psfDiffFlux`, `psfDiffFluxErr` (nJy), `visit`, flags |
| `type`, `redshift`, `t0`, `mwebv`, `template` | truth (simulations only)                      |

`hyrax_parsnip.RUBIN_DIA_SETTINGS` maps these columns for the dataset, rescales nJy to the PLAsTiCC zeropoint
(27.5), and drops flagged points.

The simulation forward-models each supernova through the **DP2 visit-detector table**: every CCD visit
covering its position gives one forced-photometry point, with noise from that visit's sky, zeropoint and seeing
(Poisson errors inflated by 1.5, as in `dp2_sims_visitdetector`). Positions are drawn uniformly over the
area covered by both DP2 visits and the DP2 DIA catalog, and Milky Way extinction comes from SFD.

- **SN Ia**: SALT3, redshift from the Frohmaier et al. (2019) volumetric rate, asymmetric Gaussian priors on
  `x1` and `c`, and `x0` from the Tripp relation (same priors as `dp2_sims_visitdetector`).
- **SN II** (II-P/L, IIn, IIb) and **SN Ib/c** (Ib, Ic, Ic-BL): one of the Vincenzi et al. (2019) templates
  per object, scaled to a Gaussian peak absolute B magnitude, with a rate ∝ (1+z)^2.7.

A DIA object only exists if the source was detected, so `simulate_catalog` keeps objects with at least 2
unsaturated points above SNR 5, and simulates in batches until each class has `N_PER_CLASS`.
The first run downloads the templates into the sncosmo cache.

In [ ]:
SEED = 1024
N_PER_CLASS = 1000  # detected objects kept per class

CLASSES = simulate_dp2.CLASSES  # class -> redshift range, templates, peak magnitude
BANDS = simulate_dp2.BANDS
CLASSES

Load the DP2 visit-detector table (about 5 million CCD visits) and the sky area to simulate:

In [ ]:
survey = simulate_dp2.load_survey(noise_scale=1.5)  # DP2 flux errors are underestimated
print(f"{len(survey.survey_info.obstable):,} CCD visits, MJD {survey.t_min:.1f}-{survey.t_max:.1f}")
print(f"Simulated area: {survey.footprint.sky_fraction * 41253:.0f} deg²")

Simulate each class and write the HATS catalog. lightcurvelynx fluxes are already in nJy, like Rubin's.
Saturated points get `psfDiffFlux_flag`; the other flags the dataset checks are all False.

In [ ]:
simulated = simulate_dp2.simulate_catalog(survey, N_PER_CLASS, seed=SEED)
simulate_dp2.write_catalog(simulated, CATALOG_PATH)
CATALOG_PATH

Check that the simulated catalog has the DP2 DIA schema: every column it shares with the real catalog has the same
type, and the only extra columns are the truth columns.

In [ ]:
print("simulation-only columns:", simulate_dp2.check_schema(CATALOG_PATH))

Open the catalog with LSDB to check what it looks like:

In [ ]:
catalog = lsdb.open_catalog(CATALOG_PATH)
frame = catalog.compute()
print(f"{len(frame)} objects, columns: {list(frame.columns)}")
print(frame["type"].value_counts().to_dict())
frame[["diaObjectId", "type", "template", "redshift", "nDiaSources", "diaObjectForcedSource"]].head()

In [ ]:
BAND_COLORS = dict(zip(BANDS, ["tab:purple", "tab:blue", "tab:green", "tab:orange", "tab:red", "tab:brown"]))


def plot_lightcurve(ax, row, title=None):
    # Forced photometry covers all of DP2, so show -30 to +100 days around the true t0.
    lc = row["diaObjectForcedSource"]
    lc = lc[(lc["midpointMjdTai"] > row["t0"] - 30) & (lc["midpointMjdTai"] < row["t0"] + 100)]
    for band in BANDS:
        sel = lc[lc["band"] == band]
        ax.errorbar(
            sel["midpointMjdTai"], sel["psfDiffFlux"], sel["psfDiffFluxErr"], fmt="o", ms=3, color=BAND_COLORS[band], label=band
        )
    ax.set_title(title or f"{row['diaObjectId']} ({row['type']}, z={row['redshift']:.2f})")
    ax.set_xlabel("MJD")


# The best-sampled object of each class.
examples = pd.concat([frame[frame["type"] == c].sort_values("nDiaSources").tail(1) for c in CLASSES])
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (_, row) in zip(axes, examples.iterrows()):
    plot_lightcurve(ax, row)
axes[0].set_ylabel("psfDiffFlux (nJy)")
axes[-1].legend(ncol=2, fontsize=8)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label in CLASSES:
    sel = frame[frame["type"] == label]
    axes[0].hist(sel["redshift"], bins=30, histtype="step", label=label)
    axes[1].hist(sel["nDiaSources"], bins=np.arange(0, 60, 2), histtype="step", label=label)
axes[0].set_xlabel("redshift")
axes[1].set_xlabel("detections (nDiaSources)")
axes[0].legend()
plt.tight_layout()

## 2. Inference with a pretrained ParSNIP model

`hyrax_parsnip.configure` points a Hyrax instance at the catalog and the ParSNIP model. With
`pretrained="plasticc"` it also sets `infer.model_weights_file` to ParSNIP's built-in PLAsTiCC
model, so `h.infer()` runs right away without a training run.

`RUBIN_DIA_SETTINGS` points the dataset at the DIA columns. The `plasticc` model needs a redshift, so here we
also give it the true redshift, and the simulated Milky Way E(B-V) for its extinction correction.

The default `band_map` maps catalog bands `ugrizy` to `lsstu`…`lssty`, the bands the `plasticc`
model knows. For another survey, set `data_set.ParsnipHATSDataset.band_map` (globally, not per data request).

In [ ]:
def configure_dia(h, pretrained, redshift_column=False, **kwargs):
    hyrax_parsnip.configure(h, CATALOG_PATH, pretrained=pretrained, **kwargs)
    for key, value in hyrax_parsnip.RUBIN_DIA_SETTINGS.items():
        h.set_config(f"data_set.ParsnipHATSDataset.{key}", value)
    h.set_config("data_set.ParsnipHATSDataset.redshift_column", redshift_column)
    h.set_config("data_set.ParsnipHATSDataset.mwebv_column", "mwebv")
    return h


h = Hyrax()
h.set_config("general.results_dir", str(RESULTS_DIR))
h.set_config("data_loader.batch_size", 64)
configure_dia(h, "plasticc", redshift_column="redshift")

h.config["data_set"]["ParsnipHATSDataset"]["band_map"]

In [ ]:
results = h.infer()

`load_predictions` turns Hyrax's results into a ParSNIP-style astropy table (one row per object),
and can join extra catalog columns such as the class label. Objects ParSNIP can't process get NaN
features instead of being dropped, so ids stay aligned. Hyrax keeps ids as strings, so `object_id`
is the `diaObjectId` as a string.

In [ ]:
labels = hyrax_parsnip.catalog_metadata(CATALOG_PATH, ["type", "redshift"], id_column="diaObjectId")
predictions = hyrax_parsnip.load_predictions(results, metadata=labels)

print(f"{len(predictions)} objects, {len(predictions.colnames)} columns")
predictions["object_id", "type", "s1", "s2", "s3", "color", "amplitude", "luminosity", "reference_time"][:5]

### Without redshifts

Real DIA objects have no spectroscopic redshift. `plasticc_photoz` predicts it from the light curve instead;
`RUBIN_DIA_SETTINGS` already sets `redshift_column = False`. This is the setup `examples/parsnip_classify_rubin_dia.py`
uses on the DP2 DIA catalog, and here we can check its redshifts against the truth:

In [ ]:
h_photoz = Hyrax()
h_photoz.set_config("general.results_dir", str(RESULTS_DIR))
h_photoz.set_config("data_loader.batch_size", 64)
configure_dia(h_photoz, "plasticc_photoz")

photoz = hyrax_parsnip.load_predictions(h_photoz.infer(), metadata=labels)
ok = np.isfinite(np.asarray(photoz["predicted_redshift"]))
true_z, pred_z = np.asarray(photoz["redshift"])[ok], np.asarray(photoz["predicted_redshift"])[ok]
dz = (pred_z - true_z) / (1 + true_z)
nmad = 1.4826 * np.median(np.abs(dz - np.median(dz)))
print(f"{ok.sum()} objects: Δz/(1+z) median {np.median(dz):+.3f}, NMAD {nmad:.3f}")

fig, ax = plt.subplots(figsize=(4.5, 4))
types = np.asarray(photoz["type"])[ok]
for label in CLASSES:
    ax.scatter(true_z[types == label], pred_z[types == label], s=6, alpha=0.5, label=label)
ax.plot([0, 0.8], [0, 0.8], "k--", lw=1)
ax.set(xlabel="true redshift", ylabel="ParSNIP predicted redshift")
ax.legend()
plt.tight_layout()

## 3. The ParSNIP latent space

ParSNIP describes each light curve with intrinsic latents `s1..s3`, plus a color, an amplitude and
a reference time. Here are the classes in the latent space of the `plasticc` model (true redshifts):

In [ ]:
types = np.asarray(predictions["type"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (x, y) in zip(axes, [("s1", "s2"), ("s1", "s3"), ("s2", "s3")]):
    for label, color in zip(CLASSES, ["tab:blue", "tab:orange", "tab:green"]):
        sel = types == label
        ax.scatter(predictions[x][sel], predictions[y][sel], s=6, alpha=0.5, color=color, label=label)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
axes[0].legend()
plt.tight_layout()

## 4. Classification

ParSNIP doesn't ship a pretrained classifier. `train_classifier` trains ParSNIP's LightGBM
classifier on the labeled predictions with K-fold cross-validation and returns out-of-sample
probabilities for every training object, which give a fair estimate of accuracy.

ParSNIP's default LightGBM `min_child_weight=1000` is tuned for PLAsTiCC-sized training sets (thousands
of objects per class). With about a thousand objects per class it prevents most splits, so we lower it here.
On a large training set, keep the default.

In [ ]:
classifier, out_of_sample = hyrax_parsnip.train_classifier(
    predictions, label_column="type", num_folds=5, min_child_weight=10.0
)

from astropy.table import join

class_names = [str(c) for c in classifier.class_names]
oos = join(out_of_sample, predictions["object_id", "type"], keys="object_id")
oos_probs = np.stack([np.asarray(oos[c]) for c in class_names], axis=1)
oos_pred = np.asarray(class_names)[oos_probs.argmax(axis=1)]
oos_true = np.asarray(oos["type"])

print(f"Out-of-sample accuracy: {np.mean(oos_pred == oos_true):.3f}")
pd.crosstab(pd.Series(oos_true, name="true"), pd.Series(oos_pred, name="predicted"))

These are realistic DP2 light curves: irregular cadence, gaps, many objects with only a few detections, and
several templates per core-collapse class, so the classes overlap. The classes are also balanced here, unlike
a real sample, where SNe Ia dominate at these depths.

Save the classifier and reload it later to classify **new** inference results without retraining.
Here we reload it and apply it to the same predictions to show the round trip.

In [ ]:
classifier_path = RESULTS_DIR / "parsnip_classifier.pkl"
classifier.write(str(classifier_path.resolve()))

reloaded = hyrax_parsnip.load_classifier(classifier_path)
probabilities = hyrax_parsnip.classify(reloaded, predictions)
probabilities[:5]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
probs_by_id = {row["object_id"]: row for row in probabilities}
for ax, (_, row) in zip(axes, examples.iterrows()):
    p = probs_by_id[str(row["diaObjectId"])]
    best = max(class_names, key=lambda c: p[c])
    plot_lightcurve(ax, row, title=f"{row['diaObjectId']}: true {row['type']}, P({best})={p[best]:.2f}")
axes[0].set_ylabel("psfDiffFlux (nJy)")
plt.tight_layout()

## 5. *(Optional)* Fine-tune ParSNIP with Hyrax

Training is optional. `h.train()` uses ParSNIP's own loss, optimizer, augmentation and learning-rate
schedule. Pass `pretrained=False` to train from scratch, or a model name to fine-tune it.
`model_weights_file=False` makes the next `h.infer()` use the weights from this training run.

A Hyrax epoch is one pass over the catalog. ParSNIP's own `fit` counts about 25,000 augmented light curves as
an epoch, so on a real catalog you'd use far more epochs than the few we run here.

In [ ]:
h_train = Hyrax()
h_train.set_config("general.results_dir", str(RESULTS_DIR))
h_train.set_config("data_loader.batch_size", 64)
h_train.set_config("train.epochs", 3)
configure_dia(h_train, "plasticc", redshift_column="redshift", groups=("train", "infer"), model_weights_file=False)

model = h_train.train()
model.final_training_metrics

Export the fine-tuned model in ParSNIP's native `.pt` format. It then works with `parsnip.load_model` or `configure(..., pretrained="path.pt")`:

In [ ]:
export_path = RESULTS_DIR / "parsnip_finetuned.pt"
model.export_parsnip(export_path)

finetuned = hyrax_parsnip.load_predictions(h_train.infer(), metadata=labels)
finetuned["object_id", "type", "s1", "s2", "s3", "color", "luminosity"][:5]

## Next steps

- Run on real data: `python examples/parsnip_classify_rubin_dia.py <DP2 DIA HATS catalog> --classifier <classifier.pkl>`.
  The simulated catalog has the same schema, so you can try the script on `data/dp2_sim_sne` first.
- For a classifier to use on real data, train it on `plasticc_photoz` predictions (no redshifts), with more
  objects per class (`N_PER_CLASS`) and realistic class fractions.
- See the `README.md` for macOS notes (CPU device, single-threaded LightGBM) and known limitations.